In [ ]:
pip install streamlit torch torchvision pillow matplotlib numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 85.0 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1


In [1]:
%%writefile app.py
import sys
sys.path.append('/content/drive/MyDrive/Project')
import streamlit as st
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
import base64

# Import Generator class - make sure it's in the correct location
from models.generator import Generator

# Function to add background image with reduced opacity
def add_bg_from_url(image_file):
    with open(image_file, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode()

    st.markdown(
        f"""
        <style>
        .stApp {{
            background-image: linear-gradient(rgba(0, 0, 0, 0.7), rgba(0, 0, 0, 0.7)), url("data:image/png;base64,{encoded_string}");
            background-size: cover;
            background-position: center;
            background-repeat: no-repeat;
            background-attachment: fixed;
        }}
        /* Remove default padding */
        .css-18e3th9 {{
            padding-top: 0rem !important;
        }}
        /* Make container more visible */
        .block-container {{
            background-color: transparent !important;
            padding: 2rem;
            margin: 0;
        }}
        /* Center content */
        .css-1kyxreq {{
            justify-content: center;
        }}
        /* Text styling */
        h1, h2, h3 {{
            color: #ffffff;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);
        }}
        p, li {{
            color: #ffffff;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.7);
        }}
        .big-font {{
            font-size: 3.5rem !important;
            font-family: "Times New Roman", Times, serif;
            color: #FFFACD;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);
            text-align: center; /* Center title text */
        }}
        .medium-font {{
            font-size: 1.8rem !important;
            font-family: "Times New Roman", Times, serif;
            color: #FFFACD;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.5);
            text-align: center; /* Center subtitle text */
        }}
        .landing-text {{
            color: #ffffff;
            font-size: 1.2rem;
            line-height: 1.8;
            text-shadow: 1px 1px 2px rgba(0,0,0,0.7);
        }}
        /* Blue content containers */
        .blue-container {{
            background-color: rgba(41, 128, 185, 0.3);
            padding: 1.5rem;
            border-radius: 0.5rem;
            border-left: 4px solid #3498DB;
            margin: 1.5rem 0;
            backdrop-filter: blur(5px);
        }}
        .highlight {{
            background-color: rgba(41, 128, 185, 0.3);
            padding: 1rem;
            border-radius: 0.5rem;
            border-left: 5px solid #3498DB;
            margin: 1rem 0;
            backdrop-filter: blur(5px);
        }}
        .footer {{
            padding: 1rem;
            text-align: center;
            color: #ffffff;
            background-color: rgba(41, 128, 185, 0.3);
            border-radius: 0.5rem;
            margin-top: 2rem;
            backdrop-filter: blur(5px);
        }}
        /* Button styling - smaller size for launch button */
        .launch-button {{
            width: 200px !important;
            margin: 0 auto;
            display: block;
        }}
        .stButton>button {{
            background-color: rgba(41, 128, 185, 0.8);
            color: white;
            padding: 0.5rem 1.5rem;
            font-size: 1.1rem;
            border-radius: 0.5rem;
            transition: all 0.3s;
            border: 1px solid #ffffff;
        }}
        .stButton>button:hover {{
            background-color: rgba(26, 82, 118, 0.9);
            box-shadow: 0 4px 8px rgba(0, 0, 0, 0.3);
            transform: translateY(-2px);
        }}
        /* More visible cards */
        .result-card {{
            background-color: rgba(41, 128, 185, 0.3);
            padding: 1.5rem;
            border-radius: 0.5rem;
            box-shadow: 0 2px 8px rgba(0, 0, 0, 0.3);
            margin: 1rem 0;
            color: #ffffff;
            backdrop-filter: blur(5px);
        }}
        .image-container {{
            display: flex;
            justify-content: center;
            background-color: rgba(41, 128, 185, 0.3);
            padding: 1rem;
            border-radius: 0.5rem;
            margin: 1rem 0;
            backdrop-filter: blur(5px);
        }}
        /* Tab styling */
        .stTabs [data-baseweb="tab-list"] {{
            gap: 24px;
            background-color: rgba(41, 128, 185, 0.3);
            padding: 10px;
            border-radius: 10px;
            backdrop-filter: blur(5px);
        }}
        .stTabs [data-baseweb="tab"] {{
            height: 50px;
            white-space: pre-wrap;
            background-color: rgba(41, 128, 185, 0.2);
            border-radius: 4px 4px 0px 0px;
            gap: 1px;
            padding-top: 10px;
            padding-bottom: 10px;
            color: white;
        }}
        .stTabs [aria-selected="true"] {{
            background-color: rgba(41, 128, 185, 0.7) !important;
            color: white !important;
        }}
        /* Make widgets more visible */
        .stSelectbox>div>div, .stFileUploader>div, .stMarkdown pre {{
            background-color: rgba(41, 128, 185, 0.3) !important;
            color: white !important;
            backdrop-filter: blur(5px);
        }}
        .st-dg {{
            color: white !important;
        }}
        /* Style metrics */
        .css-1wivap2 {{
            background-color: rgba(41, 128, 185, 0.3) !important;
            border-radius: 10px;
            padding: 10px;
            backdrop-filter: blur(5px);
        }}
        .css-50ug3q, .css-16idsys {{
            color: white !important;
        }}
        /* Style form elements */
        .stFileUploader {{
            background-color: rgba(41, 128, 185, 0.3);
            padding: 20px;
            border-radius: 10px;
            border: 1px dashed #3498DB;
            backdrop-filter: blur(5px);
        }}
        /* Style expanders */
        .streamlit-expanderHeader {{
            background-color: rgba(41, 128, 185, 0.3) !important;
            color: white !important;
            border-radius: 5px;
            backdrop-filter: blur(5px);
        }}
        .streamlit-expanderContent {{
            background-color: rgba(41, 128, 185, 0.2) !important;
            color: white !important;
            border-radius: 0 0 5px 5px;
            backdrop-filter: blur(5px);
        }}
        /* Two-column landing page layout */
        .landing-columns {{
            display: flex;
            flex-direction: row;
            gap: 2rem;
            margin-top: 2rem;
        }}
        .landing-image-column {{
            flex: 1;
            background-color: rgba(41, 128, 185, 0.3);
            border-radius: 0.8rem;
            overflow: hidden;
            margin-bottom: 1rem;
            border: 1px solid rgba(255, 255, 255, 0.3);
            backdrop-filter: blur(5px);
        }}
        .landing-content-column {{
            flex: 1;
            background-color: rgba(41, 128, 185, 0.3);
            border-radius: 0.8rem;
            padding: 2rem;
            margin-bottom: 1rem;
            backdrop-filter: blur(5px);
        }}
        /* Center buttons container */
        .center-buttons {{
            display: flex;
            justify-content: center;
            margin: 20px 0;
        }}
        /* For small screens - make columns stack */
        @media (max-width: 768px) {{
            .landing-columns {{
                flex-direction: column;
            }}
        }}
        </style>
        """,
        unsafe_allow_html=True
    )

# Card component for nicer UI sections
def card(title, content):
    st.markdown(f"""
    <div class="result-card">
        <h3>{title}</h3>
        {content}
    </div>
    """, unsafe_allow_html=True)

class DRProgressionSystem:
    def __init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        self.num_classes = 5
        self.class_names = ['mild', 'moderate', 'noDr', 'pdr', 'severe']
        self.classifier = self._load_classifier()
        self.generators = self._load_generators()

        self.classify_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

        self.gan_transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])

        self.inverse_transform = transforms.Normalize((-1, -1, -1), (2, 2, 2))

    def _load_classifier(self):
        model = efficientnet_b0()
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(model.classifier[1].in_features, self.num_classes)
        )

        # Model path - update this based on your deployment environment
        classifier_path = "/content/drive/MyDrive/Project/models/efficientnetb0_dr_final.pth"
        model.load_state_dict(torch.load(classifier_path, map_location=self.device))
        model.to(self.device)
        model.eval()
        print("EfficientNet-B0 classifier loaded.")
        return model


    def _load_generators(self):
        generators = {}

        # Fixed generator paths - using A2B instead of B2A
        paths = {
            "Mild_to_Moderate": "/content/drive/MyDrive/Project/models/Mild_to_Moderate/netG_A2B_final.pth",
            "Moderate_to_Severe": "/content/drive/MyDrive/Project/models/Moderate_to_Severe/netG_A2B_final.pth"
        }

        for key, path in paths.items():
            gen = Generator(3, 3).to(self.device)
            gen.load_state_dict(torch.load(path, map_location=self.device))
            gen.eval()
            generators[key] = gen
            print(f"{key} generator loaded.")

        return generators

    def _interpolate_stages(self, source_img, source_stage, target_stage, steps=3):
        key = f"{source_stage}_to_{target_stage}"
        if key not in self.generators:
            raise ValueError(f"No generator found for {key}")

        with torch.no_grad():
            target_img = self.generators[key](source_img)

        interpolated_images = []
        for i in range(1, steps + 1):
            alpha = i / steps
            interp_img = (1 - alpha) * source_img + alpha * target_img
            interpolated_images.append(interp_img)

        return interpolated_images


    def process_image(self, image_path):
        img = Image.open(image_path).convert('RGB')

        # Classification
        classify_img = self.classify_transform(img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            outputs = self.classifier(classify_img)
            _, predicted = torch.max(outputs, 1)
        stage = self.class_names[predicted.item()]
        print(f"Image classified as: {stage}")

        # Based on stage, generate progression if applicable
        if stage == "noDr":
            return {"stage": stage, "message": "Healthy eye. No DR.", "progression_images": None}
        elif stage == "pdr":
            return {"stage": stage, "message": "Final DR stage. Immediate care needed.", "progression_images": None}
        else:
            gan_img = self.gan_transform(img).unsqueeze(0).to(self.device)
            if stage == "mild":
                images = self._interpolate_stages(gan_img, "Mild", "Moderate", steps=3)
            elif stage == "moderate":
                images = self._interpolate_stages(gan_img, "Moderate", "Severe", steps=3)
            else:
                return {"stage": stage, "message": "Severe DR. Close to PDR.", "progression_images": None}

            # Apply inverse transform and return
            progression = [self.inverse_transform(img.squeeze(0)).cpu() for img in images]
            return {
                "stage": stage,
                "message": f"{stage} DR detected. Projected progression shown below.",
                "progression_images": progression
            }

# Streamlit app
# Inside the main() function, update the landing page code:

def main():
    st.set_page_config(
        page_title="DR Progression Analyzer",
        page_icon="👁️",
        layout="wide"
    )

    # Get the directory of the current script
    current_dir = os.path.dirname(os.path.abspath(__file__))

    # Add the background - use relative path
    bg_path = "/content/drive/MyDrive/static/images/bg_last.jpg"
    add_bg_from_url(bg_path)

    # Check if we're on the landing page or main application
    if 'started' not in st.session_state:
        st.session_state.started = False

    if not st.session_state.started:
        # LANDING PAGE - With two-column layout

        # Title
        st.markdown("""
        <div class="blue-container">
            <h1 class='big-font'>DRishti</h1>
            <h2 class='medium-font'>Advanced DR Progression Analysis System</h2>
        </div>
        """, unsafe_allow_html=True)

        # Two-column layout using Streamlit columns
        col1, col2 = st.columns(2)

        with col1:
            # Eye image with proper path - fall back to URL if file not found
            eye_image_path = os.path.join("/content/drive/MyDrive/static/images/bg_last.jpg", "eye_image.jpg")
            try:
                if os.path.exists(eye_image_path):
                    st.image(eye_image_path, width=None)
                else:
                    # Fallback to a placeholder or URL
                    st.image("/content/drive/MyDrive/static/Screenshot 2025-04-19 111005.png", width=1000)
            except Exception as e:
                st.error(f"Could not load image: {str(e)}")

        with col2:
            st.markdown("""
            <div class="blue-container">
                <h3 style="text-align:center;">Diabetic Retinopathy Simulator</h3>
                <p style="text-align:center;">
                    Early detection and progression analysis of diabetic retinopathy can prevent vision loss.
                    Our AI-powered tool provides accurate detection and visualization of potential disease progression.
                </p>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("""
            <div class="blue-container">
                <h3 style="text-align:center;">Key Statistics</h3>
                <div style="display:flex; justify-content:space-between; margin-top:1rem;">
                    <div style="text-align:center; padding:0.5rem;">
                        <h4>34.6%</h4>
                        <p>Global Prevalence<br>of diabetic patients</p>
                    </div>
                    <div style="text-align:center; padding:0.5rem;">
                        <h4>#1</h4>
                        <p>Leading Cause<br>of blindness in working-age adults</p>
                    </div>
                    <div style="text-align:center; padding:0.5rem;">
                        <h4>95%</h4>
                        <p>Vision Loss<br>preventable with early detection</p>
                    </div>
                </div>
            </div>
            """, unsafe_allow_html=True)

        # Hidden button that Streamlit can detect (workaround for custom HTML button)
        col1, col2, col3 = st.columns([1, 1, 1])
        with col2:
            if st.button("Launch Analysis Tool", key="launch-btn-hidden", help="Click to start analyzing retinal images"):
                st.session_state.started = True

        # Footer
        st.markdown("<div class='footer'>© 2025 Diabetic Retinopathy Progression Analyzer</div>", unsafe_allow_html=True)

    else:
        # MAIN APPLICATION PAGE
        st.markdown("<h1>Diabetic Retinopathy Progression Analyzer</h1>", unsafe_allow_html=True)

        # Create tabs for different sections (removed Technical Info tab)
        tab1, tab2 = st.tabs(["Analysis Tool", "About DR"])

        with tab1:
            # Initialize system when app starts
            if 'system' not in st.session_state:
                with st.spinner("Loading AI models... This may take a moment."):
                    try:
                        st.session_state.system = DRProgressionSystem()
                        st.success("Models loaded successfully!")
                    except Exception as e:
                        st.error(f"Error loading models: {str(e)}")
                        st.error("Please check your model paths and file structure.")
                        return

            # Create nice UI for file upload with blue container
            st.markdown("""
            <div class="blue-container" style="text-align: center; margin-bottom: 20px;">
                <h3>Upload Retinal Image</h3>
                <p>Select a high-quality fundus image for analysis</p>
            </div>
            """, unsafe_allow_html=True)

            # File uploader
            uploaded_file = st.file_uploader("", type=["jpg", "jpeg", "png"])

            if uploaded_file is not None:
                # Display uploaded image
                image = Image.open(uploaded_file)

                # Create columns for layout
                col1, col2 = st.columns([1, 2])

                with col1:
                    st.markdown("<div class='image-container'>", unsafe_allow_html=True)
                    st.image(image, caption="Uploaded Retinal Image", use_container_width=True)
                    st.markdown("</div>", unsafe_allow_html=True)

                # Center the analyze button
                st.markdown("<div class='center-buttons'>", unsafe_allow_html=True)
                analyze_button = st.button("Analyze Image", key="analyze_btn")
                st.markdown("</div>", unsafe_allow_html=True)

                if analyze_button:
                    # Save temp file for processing
                    temp_path = "temp_image.jpg"
                    image.save(temp_path)

                    try:
                        with st.spinner("Analyzing retinal image..."):
                            # Add artificial delay for effect
                            import time
                            time.sleep(1)

                            # Process image using the loaded system
                            result = st.session_state.system.process_image(temp_path)

                        # Display results
                        stage = result["stage"]
                        message = result["message"]

                        # Style according to severity
                        severity_colors = {
                            "noDr": "green",
                            "mild": "orange",
                            "moderate": "orange",
                            "severe": "red",
                            "pdr": "darkred"
                        }
                        severity_icons = {
                            "noDr": "✅",
                            "mild": "⚠️",
                            "moderate": "⚠️",
                            "severe": "🚨",
                            "pdr": "🚨"
                        }

                        # Format stage name properly
                        stage_display = {
                            "noDr": "No DR",
                            "mild": "Mild NPDR",
                            "moderate": "Moderate NPDR",
                            "severe": "Severe NPDR",
                            "pdr": "Proliferative DR"
                        }

                        st.markdown(f"""
                        <div class="blue-container" style="border-left: 5px solid {severity_colors[stage]};">
                            <h2 style="margin: 0;">{severity_icons[stage]} Diagnosis: {stage_display[stage]}</h2>
                        </div>
                        """, unsafe_allow_html=True)

                        st.markdown(f"**Assessment:** {message}")

                        # Show recommendations based on stage in a blue container
                        st.subheader("Clinical Recommendations:")

                        if stage == "noDr":
                            st.markdown("""
                <div class="blue-container">
                    <ul>
                        <li>✅ Continue regular annual screenings</li>
                        <li>✅ Maintain good blood glucose control</li>
                        <li>✅ Regular eye exams as recommended by your doctor</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)
                        elif stage == "mild":
                            st.markdown("""
                <div class="blue-container">
                    <ul>
                        <li>⚠️ Schedule follow-up in 9-12 months</li>
                        <li>⚠️ Improve glycemic control</li>
                        <li>⚠️ Monitor for any vision changes</li>
                        <li>⚠️ Regular blood pressure monitoring</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)
                        elif stage == "moderate":
                            st.markdown("""
                <div class="blue-container">
                    <ul>
                        <li>⚠️ Schedule follow-up in 6 months</li>
                        <li>⚠️ Strict glycemic control essential</li>
                        <li>⚠️ Consider referral to retina specialist</li>
                        <li>⚠️ Control blood pressure and lipid levels</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)
                        elif stage == "severe":
                            st.markdown("""
                <div class="blue-container">
                    <ul>
                        <li>🚨 Urgent referral to ophthalmologist</li>
                        <li>🚨 May require laser treatment soon</li>
                        <li>🚨 Very close monitoring required</li>
                        <li>🚨 Aggressive management of diabetes</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)
                        else:  # pdr
                            st.markdown("""
                <div class="blue-container">
                    <ul>
                        <li>🚨 IMMEDIATE specialist care needed</li>
                        <li>🚨 High risk of vision loss</li>
                        <li>🚨 Likely requires laser/surgical intervention</li>
                        <li>🚨 Intensive diabetes management required</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)

                        # Display progression visualization if available
                        progression_images = result["progression_images"]
                        if progression_images is not None:
                            st.subheader("Projected Disease Progression")

                            # Set figure with transparent background
                            fig, axes = plt.subplots(1, 4, figsize=(15, 4))
                            fig.patch.set_facecolor('none')

                            # Original image
                            axes[0].imshow(image)
                            axes[0].set_title(f"Current: {stage_display[stage]}", fontsize=12, fontweight='bold', color='white')
                            axes[0].axis("off")

                            # Progression images
                            for i, img in enumerate(progression_images):
                                img_np = img.numpy().transpose(1, 2, 0)
                                img_np = np.clip(img_np, 0, 1)
                                axes[i+1].imshow(img_np)

                                # Set appropriate titles
                                if stage == "mild":
                                    next_stage = "Moderate NPDR"
                                elif stage == "moderate":
                                    next_stage = "Severe NPDR"
                                else:
                                    next_stage = "PDR"

                                axes[i+1].set_title(f"Step {i+1} toward {next_stage}", fontsize=11, color='white')
                                axes[i+1].axis("off")

                            plt.tight_layout()

                            # Make the figure background transparent
                            for ax in axes:
                                ax.set_facecolor('none')

                            # Convert matplotlib figure to image
                            buf = BytesIO()
                            fig.savefig(buf, format="png", dpi=150, bbox_inches='tight', transparent=True)
                            buf.seek(0)

                            # Display the figure
                            with col2:
                                st.image(buf, use_container_width=True)
                                st.markdown("""
                                <div class="blue-container" style="font-style: italic;">
                                    This visualization shows AI-generated projection of potential disease progression if left untreated.
                                    Early intervention can prevent this progression.
                                </div>
                                """, unsafe_allow_html=True)

                        # Clean up temp file
                        os.remove(temp_path)

                    except Exception as e:
                        st.error(f"Error during analysis: {str(e)}")

        with tab2:
            st.markdown("""
            <div class="blue-container">
                <h2>About Diabetic Retinopathy</h2>
            </div>
            """, unsafe_allow_html=True)

            # Create two columns layout
            col1, col2 = st.columns([3, 2])

            with col1:
                st.markdown("""
                <div class="blue-container">
                    Diabetic retinopathy (DR) is a diabetes complication that affects the eyes. It's caused by damage to the blood vessels
                    in the retina (the light-sensitive tissue at the back of the eye).
                </div>

                <div class="blue-container">
                    <h3>Stages of Diabetic Retinopathy:</h3>
                </div>
                """, unsafe_allow_html=True)

                # Use expanders for each stage for better organization
                with st.expander("No DR", expanded=True):
                    st.markdown("Healthy eye with no detectable signs of diabetic retinopathy.")

                with st.expander("Mild Non-Proliferative DR (NPDR)"):
                    st.markdown("Early stage with small areas of balloon-like swelling in the retina's tiny blood vessels.")

                with st.expander("Moderate NPDR"):
                    st.markdown("As the disease progresses, more blood vessels are blocked, causing noticeable changes to the retina.")

                with st.expander("Severe NPDR"):
                    st.markdown("Many more blood vessels are blocked, depriving blood supply to areas of the retina. The retina signals the body to grow new blood vessels.")

                with st.expander("Proliferative DR (PDR)"):
                    st.markdown("The most advanced stage where new, abnormal blood vessels grow. These vessels are fragile and can leak blood, causing severe vision problems.")

            with col2:
                st.markdown("""
                <div class="blue-container">
                    <h3>Risk Factors:</h3>
                    <ul>
                        <li>Duration of diabetes</li>
                        <li>Poor blood sugar control</li>
                        <li>High blood pressure</li>
                        <li>High cholesterol</li>
                        <li>Pregnancy</li>
                        <li>Smoking</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)

                st.markdown("""
                <div class="blue-container">
                    <h3>Prevention and Management:</h3>
                    <ul>
                        <li>Regular eye examinations</li>
                        <li>Good blood glucose control</li>
                        <li>Blood pressure management</li>
                        <li>Prompt treatment when indicated</li>
                    </ul>
                </div>
                """, unsafe_allow_html=True)

            # Images showing different stages
            st.markdown("<div class='image-container'>", unsafe_allow_html=True)
            st.image("https://www.thindeyehospital.org/wp-content/uploads/2022/08/diabetic-retinopathy.jpg",
                    caption="Illustration of diabetic retinopathy stages")
            st.markdown("</div>", unsafe_allow_html=True)

        # "Go back to home page" button - smaller size and centered
        st.markdown("<div class='center-buttons'>", unsafe_allow_html=True)
        if st.button("Back to Home", key="home_button"):
            st.session_state.started = False
        st.markdown("</div>", unsafe_allow_html=True)

if __name__ == "__main__":
    main()

Writing app.py


In [2]:
!pip install pyngrok


In [3]:
from pyngrok import ngrok
ngrok.kill()


In [4]:
from pyngrok import ngrok
ngrok.set_auth_token("2vPFVgqOZ0ASM5Ed7wfSpJiROLA_212TMrxTMgfVMKDGgAGcx")


In [5]:
# Install dependencies (only once)
!pip install streamlit pyngrok --quiet

# Kill any running Streamlit process
!killall -9 streamlit > /dev/null 2>&1

# Kill any existing ngrok process (IMPORTANT)
!pkill ngrok > /dev/null 2>&1

# Start Streamlit in background
!nohup streamlit run app.py --server.port 8501 > /dev/null 2>&1 &

# Give Streamlit time to start
import time
time.sleep(5)

# Setup ngrok cleanly
from pyngrok import ngrok

ngrok.kill()  # ensures no old tunnel remains

public_url = ngrok.connect(8501)
print("✅ Streamlit app is running at:", public_url)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.0 MB/s eta 0:00:00
✅ Streamlit app is running at: NgrokTunnel: "https://532f-34-182-178-241.ngrok-free.app" -> "http://localhost:8501"


!streamlit run /content/drive/MyDrive/Project/models/app.py